In [1]:
%pip install -U pip
%pip install pandas numpy scikit-learn tqdm

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import mindspore as ms
from mindspore import context, nn, ops, Tensor

from sklearn.metrics import roc_auc_score, log_loss

# GPU
context.set_context(mode=context.GRAPH_MODE, device_target="GPU")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ms.set_seed(SEED)

DATA_PATH = "assistments_2009_2010.csv"
assert os.path.exists(DATA_PATH), f"Missing file: {DATA_PATH}"
print("MindSpore:", ms.__version__)
print("Device:", ms.get_context("device_target"))
print("DATA_PATH:", DATA_PATH)

MindSpore: 1.8.0
Device: GPU
DATA_PATH: assistments_2009_2010.csv


Load dataset + select usable columns

We will use:

user_id (sequence key)

correct (label)

list_skill_ids (preferred) else problem_id (fallback)

order_id for ordering (fallback to row order if needed)

In [3]:
df = pd.read_csv(DATA_PATH)

# Minimal required columns (some may exist; we handle gracefully)
required_any = ["user_id", "correct"]
for c in required_any:
    assert c in df.columns, f"Missing required column: {c}"

# Create an event order column
if "order_id" in df.columns:
    # order_id sometimes looks like scientific notation; keep numeric
    df["event_order"] = pd.to_numeric(df["order_id"], errors="coerce")
else:
    df["event_order"] = np.arange(len(df), dtype=np.int64)

# Skill source
has_list_skill = "list_skill_ids" in df.columns
has_problem = "problem_id" in df.columns
assert has_list_skill or has_problem, "Need list_skill_ids or problem_id for skill."

def pick_first_skill(x):
    # list_skill_ids sometimes empty or NaN
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    # Sometimes it's "279" or "279, 10" or "279;10"
    for sep in [",", ";", "|"]:
        if sep in s:
            s = s.split(sep)[0].strip()
            break
    try:
        return int(float(s))
    except:
        return np.nan

if has_list_skill:
    df["skill_raw"] = df["list_skill_ids"].apply(pick_first_skill)
else:
    df["skill_raw"] = np.nan

# Fallback to problem_id when skill missing
if has_problem:
    df["problem_id_num"] = pd.to_numeric(df["problem_id"], errors="coerce")
    df["skill_raw"] = df["skill_raw"].fillna(df["problem_id_num"])
else:
    df["skill_raw"] = df["skill_raw"]

# Clean correct
df["correct"] = pd.to_numeric(df["correct"], errors="coerce").fillna(0).astype(int)
df["correct"] = df["correct"].clip(0, 1)

# Drop rows missing essentials
df = df.dropna(subset=["user_id", "skill_raw", "event_order"]).copy()
df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce")
df = df.dropna(subset=["user_id"]).copy()
df["user_id"] = df["user_id"].astype(int)
df["skill_raw"] = df["skill_raw"].astype(int)

# Sort within user
df = df.sort_values(["user_id", "event_order"]).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Unique users:", df["user_id"].nunique())
print(df[["user_id","skill_raw","correct","event_order"]].head())


Rows after cleaning: 1011079
Unique users: 8519
   user_id  skill_raw  correct  event_order
0    21432         46        1     20235577
1    21825         10        0     21616385
2    21825      48882        1     21616426
3    21825      48883        0     21616431
4    21825      48884        1     21616443


Map skills to contiguous IDs + build per-user sequences

In [4]:
# Skill vocabulary
unique_skills = df["skill_raw"].unique()
skill_to_idx = {s:i for i,s in enumerate(unique_skills)}
idx_to_skill = {i:s for s,i in skill_to_idx.items()}
K = len(skill_to_idx)
print("Num skills (K):", K)

# Encode skill index
df["skill_idx"] = df["skill_raw"].map(skill_to_idx).astype(int)

# Build sequences per user: list of (skill_idx, correct)
user_seqs = []
user_ids = []

for uid, g in df.groupby("user_id", sort=False):
    skills = g["skill_idx"].to_numpy(np.int32)
    corr = g["correct"].to_numpy(np.int32)
    if len(skills) < 3:
        continue  # too short to be useful
    user_ids.append(uid)
    user_seqs.append((skills, corr))

lengths = np.array([len(s[0]) for s in user_seqs])
print("Sequences kept:", len(user_seqs))
print("Min/Median/Max length:", lengths.min(), np.median(lengths), lengths.max())


Num skills (K): 22570
Sequences kept: 8213
Min/Median/Max length: 3 42.0 2250


Train/Val/Test split by user (no leakage)

In [5]:
uids = np.array(user_ids)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(uids))

n = len(uids)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train+n_val]
test_idx = perm[n_train+n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs   = [user_seqs[i] for i in val_idx]
test_seqs  = [user_seqs[i] for i in test_idx]

print("Train/Val/Test users:", len(train_seqs), len(val_seqs), len(test_seqs))


Train/Val/Test users: 6570 821 822


Windowing + dataset generator (DKT-standard)

We create training samples as windows of length MAX_LEN.
Inputs are interaction tokens: x = skill + correct*K (vocab size 2K)

Targets per timestep:

next_skill (shifted skills)

next_correct (shifted correctness)

mask to ignore padding

In [6]:
MAX_LEN = 50
STRIDE  = 25   # overlap; set = MAX_LEN for non-overlap
BATCH_SIZE = 32

PAD_X = 0           # reserved token for padding
PAD_SKILL = -1      # padding marker for next_skill

def make_windows(skills, corr, max_len, stride):
    L = len(skills)
    out = []
    start = 0
    while start < L:
        end = min(start + max_len, L)
        out.append((skills[start:end], corr[start:end]))
        if end == L:
            break
        start += stride
    return out

def pad_seq(arr, max_len, pad_val):
    out = np.full((max_len,), pad_val, dtype=np.int32)
    out[:len(arr)] = arr
    return out

def gen_samples(seqs, max_len):
    """
    yields:
      x_tok      (T,) int32 tokens in [0..2K] where 0 is PAD, real tokens start at 1
      next_skill (T,) int32 in [0..K-1] or -1 for PAD
      next_corr  (T,) int32 in {0,1}
      mask       (T,) float32 in {0,1}
    """
    for skills, corr in seqs:
        for ws, wc in make_windows(skills, corr, max_len, STRIDE):
            if len(ws) < 2:
                continue

            # Tokenize interactions: token = skill + correct*K + 1
            # +1 ensures 0 is PAD only
            x_tok = (ws + wc * K + 1).astype(np.int32)
            x_tok = pad_seq(x_tok, max_len, pad_val=PAD_X)

            # targets (shifted by one)
            next_skill = pad_seq(ws[1:].astype(np.int32), max_len, pad_val=PAD_SKILL)
            next_corr  = pad_seq(wc[1:].astype(np.int32), max_len, pad_val=0)

            mask = (next_skill != PAD_SKILL).astype(np.float32)

            yield x_tok, next_skill, next_corr, mask

# sanity
tmp = next(gen_samples(train_seqs[:1], MAX_LEN))
print([a.shape for a in tmp], "mask sum:", tmp[-1].sum())

[(50,), (50,), (50,), (50,)] mask sum: 9.0


In [7]:
import mindspore.dataset as ds

ds.config.set_num_parallel_workers(1)
ds.config.set_prefetch_size(2)
ds.config.set_enable_shared_mem(False)

def build_dataset(seqs, batch_size, max_len, shuffle=True):
    columns = ["x_tok", "next_skill", "next_corr", "mask"]
    dataset = ds.GeneratorDataset(
        source=lambda: gen_samples(seqs, max_len),
        column_names=columns,
        shuffle=shuffle,
        num_parallel_workers=1,
        python_multiprocessing=False,  # critical for CUDA stability
    )
    dataset = dataset.batch(batch_size, drop_remainder=True)
    return dataset

train_ds = build_dataset(train_seqs, BATCH_SIZE, MAX_LEN, shuffle=True)
val_ds   = build_dataset(val_seqs,   BATCH_SIZE, MAX_LEN, shuffle=False)
test_ds  = build_dataset(test_seqs,  BATCH_SIZE, MAX_LEN, shuffle=False)

print("Train batches:", train_ds.get_dataset_size())

Train batches: 990


Standard DKT model (per-timestep outputs)

Outputs logits for each timestep and each skill: (B, T, K)

In [8]:
class DKTNet(nn.Cell):
    def __init__(self, num_skills, emb_dim=64, hidden_dim=128, dropout=0.0):
        super().__init__()
        self.K = num_skills
        self.vocab = 2 * num_skills + 1   # +1 for PAD=0
        self.emb = nn.Embedding(self.vocab, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True, dropout=dropout)
        self.fc = nn.Dense(hidden_dim, num_skills)

    def construct(self, x_tok):
        x = self.emb(x_tok)
        h, _ = self.gru(x)
        logits = self.fc(h)
        return logits

model = DKTNet(K, emb_dim=64, hidden_dim=128, dropout=0.0)
print(model)

DKTNet<
  (emb): Embedding<vocab_size=45141, embedding_size=64, use_one_hot=False, embedding_table=Parameter (name=emb.embedding_table, shape=(45141, 64), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
  (gru): GRU<
    (rnn): _DynamicGRUCPUGPU<>
    (dropout_op): Dropout<keep_prob=1.0>
    >
  (fc): Dense<input_channels=128, output_channels=22570, has_bias=True>
  >


Correct DKT loss (gather next_skill prob → BCE, masked)

We do:

probs = sigmoid(logits) → (B,T,K)

gather p = probs[b,t,next_skill[b,t]]

compute BCE vs next_corr

multiply by mask and average over valid targets

In [9]:
sigmoid = ops.Sigmoid()
gather = ops.GatherD()
eps = 1e-7

class DKTLossCell(nn.Cell):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def construct(self, x_tok, next_skill, next_corr, mask):
        logits = self.net(x_tok)       # (B,T,K)
        probs = sigmoid(logits)        # (B,T,K)

        # make next_skill safe for gather (replace -1 with 0)
        safe_skill = ops.maximum(next_skill, Tensor(0, ms.int32))
        idx = ops.ExpandDims()(safe_skill, -1)         # (B,T,1)
        p = gather(probs, 2, idx).squeeze(-1)          # (B,T)

        y = next_corr.astype(ms.float32)
        p = ops.clip_by_value(p, eps, 1 - eps)

        bce = -(y * ops.log(p) + (1 - y) * ops.log(1 - p))
        loss = (bce * mask).sum() / (mask.sum() + eps)
        return loss

loss_cell = DKTLossCell(model)
optimizer = nn.Adam(model.trainable_params(), learning_rate=1e-3)
train_step = nn.TrainOneStepCell(loss_cell, optimizer)
train_step.set_train()


TrainOneStepCell<
  (network): DKTLossCell<
    (net): DKTNet<
      (emb): Embedding<vocab_size=45141, embedding_size=64, use_one_hot=False, embedding_table=Parameter (name=net.emb.embedding_table, shape=(45141, 64), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
      (gru): GRU<
        (rnn): _DynamicGRUCPUGPU<>
        (dropout_op): Dropout<keep_prob=1.0>
        >
      (fc): Dense<input_channels=128, output_channels=22570, has_bias=True>
      >
    >
  (optimizer): Adam<>
  >

In [10]:
batch = next(train_ds.create_dict_iterator())
x_tok = batch["x_tok"]
print("x_tok shape:", x_tok.shape)

# forward-only (no training) to confirm CUDA path is stable
model.set_train(False)
_ = model(x_tok)
print("Forward pass OK on GPU")

x_tok shape: (32, 50)
Forward pass OK on GPU


Training + validation loop (AUC, accuracy, logloss)

In [11]:
def eval_epoch(net, dataset, max_batches=None):
    net.set_train(False)
    all_p, all_y = [], []

    for bi, batch in enumerate(dataset.create_dict_iterator()):
        if max_batches and bi >= max_batches:
            break

        x_tok = batch["x_tok"]
        ns = batch["next_skill"].asnumpy().astype(np.int32)          # (B,T)
        y  = batch["next_corr"].asnumpy().astype(np.float32)         # (B,T)
        m  = batch["mask"].asnumpy().astype(np.float32)              # (B,T)

        probs = sigmoid(net(x_tok)).asnumpy().astype(np.float32)     # (B,T,K)

        ns_clip = np.clip(ns, 0, K - 1)
        p = np.take_along_axis(probs, ns_clip[..., None], axis=2).squeeze(2)  # (B,T)

        valid = m > 0
        all_p.append(p[valid])
        all_y.append(y[valid])

    if not all_p:
        return {"auc": np.nan, "acc": np.nan, "logloss": np.nan}

    p = np.concatenate(all_p)
    y = np.concatenate(all_y)

    auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
    acc = float(((p >= 0.5) == (y >= 0.5)).mean())
    ll  = float(log_loss(y, np.clip(p, 1e-7, 1-1e-7)))

    return {"auc": float(auc), "acc": acc, "logloss": ll}


In [13]:
import os

CKPT_DIR = "/workspace/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

BEST_CKPT = os.path.join(CKPT_DIR, "best_dkt.ckpt")
print("Saving checkpoints to:", BEST_CKPT)


# BEST_CKPT = "/workspace/dkt/best_dkt.ckpt"
best_auc = -1.0
patience = 2
bad = 0

EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.set_train(True)
    losses = []

    for batch in tqdm(train_ds.create_dict_iterator(), total=train_ds.get_dataset_size(), desc=f"Epoch {epoch}"):
        loss = train_step(batch["x_tok"], batch["next_skill"], batch["next_corr"], batch["mask"])
        losses.append(float(loss.asnumpy()))

    val_metrics = eval_epoch(model, val_ds, max_batches=None)
    print(f"\nEpoch {epoch} | train_loss={np.mean(losses):.4f} | val_auc={val_metrics['auc']:.4f} | "
          f"val_acc={val_metrics['acc']:.4f} | val_logloss={val_metrics['logloss']:.4f}\n")

    if val_metrics["auc"] > best_auc + 1e-4:
        best_auc = val_metrics["auc"]
        bad = 0
        ms.save_checkpoint(model, BEST_CKPT)
        print(f"Saved best checkpoint to {BEST_CKPT} (auc={best_auc:.4f})")
    else:
        bad += 1
        if bad >= patience:
            print(f"Early stopping (best auc={best_auc:.4f})")
            break

# load best for test
ms.load_checkpoint(BEST_CKPT, net=model)
test_metrics = eval_epoch(model, test_ds, max_batches=None)
print("TEST(best):", test_metrics)

Saving checkpoints to: /workspace/checkpoints/best_dkt.ckpt


Epoch 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 990/990 [00:14<00:00, 67.59it/s]



Epoch 1 | train_loss=0.4947 | val_auc=0.7946 | val_acc=0.7417 | val_logloss=0.5120

Saved best checkpoint to /workspace/checkpoints/best_dkt.ckpt (auc=0.7946)


Epoch 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 990/990 [00:14<00:00, 67.09it/s]



Epoch 2 | train_loss=0.4671 | val_auc=0.7940 | val_acc=0.7409 | val_logloss=0.5175



Epoch 3: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 990/990 [00:14<00:00, 67.30it/s]



Epoch 3 | train_loss=0.4403 | val_auc=0.7908 | val_acc=0.7394 | val_logloss=0.5268

Early stopping (best auc=0.7946)
TEST(best): {'auc': 0.78979559009409, 'acc': 0.7260684201035575, 'logloss': 0.5307216376440007}


Final test evaluation

In [14]:
test_metrics = eval_epoch(model, test_ds, max_batches=None)
print("TEST:", test_metrics)

TEST: {'auc': 0.78979559009409, 'acc': 0.7260684201035575, 'logloss': 0.5307216376440007}


Inference: predict next correctness for a student

Given a student history of (skill_idx, correct) interactions, predict P(correct) for a candidate next skill.

In [15]:
def predict_next_correct(net, history_skills, history_corr, next_skill_idx, max_len=MAX_LEN):
    net.set_train(False)
    hs = np.asarray(history_skills, dtype=np.int32)
    hc = np.asarray(history_corr, dtype=np.int32)

    # build tokens
    x_tok = hs + hc * K
    x_tok = x_tok[-max_len:]  # truncate
    x_tok = np.pad(x_tok, (0, max_len - len(x_tok)), constant_values=0)

    x_tok = Tensor(x_tok.reshape(1, -1), ms.int32)
    logits = net(x_tok)               # (1,T,K)
    probs = sigmoid(logits).asnumpy() # (1,T,K)

    # Use last timestep output as current knowledge state
    p = float(probs[0, -1, next_skill_idx])
    return p

# Example: take first test user sequence and predict on its next step
skills, corr = test_seqs[0]
p = predict_next_correct(model, skills[:-1], corr[:-1], next_skill_idx=int(skills[-1]))
print("Predicted P(correct next):", p, "| true:", int(corr[-1]))


Predicted P(correct next): 0.6997194886207581 | true: 1
